# ML-03 ÔÇö Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haideriqbal499/Flyrank_Ml_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

Map the capstone question onto the ML loop ÔÇö task type, target, metric, unit of analysis ÔÇö *before* training anything.

> Skills loaded for this card: `framing-ml-problems` + `flyrank/flyrank-data`.

## 0. Setup (Google Colab)

Run this first so the starter CSV is on disk.

In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/haideriqbal499/Flyrank_Ml_internship"
REPO_DIR = "Flyrank_Ml_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found ÔÇö clone/setup failed"
print("Starter data found. Ready to map Lane 2 onto the ML loop.")

Working dir: C:\Users\Windows 11\Flyrank_Ml_internship
Starter data found. Ready to map Lane 2 onto the ML loop.


## 1. My lane as an ML task (type)

**Lane 2 ÔåÆ task type: ranking / scoring**

The business question is "which pages first?" ÔÇö that is a **priority ranking**, not a yes/no classification by itself. The model (or scored baseline) assigns each content item a **review-priority score**. Classification can sit underneath later (e.g. "likely to keep declining"), but the deliverable editors use is an ordered queue with reason codes.

| Piece | Choice |
|---|---|
| Lane | 2 ÔÇö Refresh / Content Opportunity Scoring |
| ML task type | Ranking / scoring |
| Output | Ranked action queue + suggested action (refresh / expand / protect / prune / monitor) |

In [2]:
TASK_TYPE = "ranking / scoring"
LANE = "Lane 2: Refresh / Content Opportunity Scoring"
OUTPUT = "ranked review queue with reason codes + suggested action"
print(f"Lane: {LANE}")
print(f"Task type: {TASK_TYPE}")
print(f"Editor-facing output: {OUTPUT}")

Lane: Lane 2: Refresh / Content Opportunity Scoring
Task type: ranking / scoring
Editor-facing output: ranked review queue with reason codes + suggested action


## 2. Target or proxy

**Primary target (preferred, warehouse path):** an *observed* future outcome ÔÇö e.g. whether a visible page loses clicks/impressions in the **next 30 days** after a decision date built from prior-window features. That label is measured in the world, not copied from a product rule.

**Starter-data proxy (what we can sketch today):** the snapshot only gives one trailing-90d window, so a true pastÔåÆfuture label is not available yet. For framing and baseline practice we use a **review-priority proxy**: pages that already look like triage candidates (high demand + downward impression trend). Important honesty check:

- `trend_direction` / `trend_pct` / `is_declining_label`-style fields are **derived snapshot labels** ÔÇö useful to *sketch* a target column, **not** legal model features later.
- When the warehouse daily table is in play, replace this proxy with a label from a **later time window** so the model learns an outcome, not a hand-written rule.

In [3]:
TARGET_NAME = "needs_refresh_review"  # starter PROXY only
TARGET_SOURCE = (
    "PROXY today: high demand (impressions_90d >= 500) AND trend_direction == 'down'. "
    "Later (warehouse): OBSERVED next-30d decline from prior-90d features."
)
print(f"Target / proxy column name: {TARGET_NAME}")
print(f"Where the label comes from: {TARGET_SOURCE}")
print("Do NOT feed trend_direction / trend_pct back in as features ÔÇö label trap.")

Target / proxy column name: needs_refresh_review
Where the label comes from: PROXY today: high demand (impressions_90d >= 500) AND trend_direction == 'down'. Later (warehouse): OBSERVED next-30d decline from prior-90d features.
Do NOT feed trend_direction / trend_pct back in as features ÔÇö label trap.


## 3. Success metric

**Primary metric: precision@K** (default K = 20 and K = 50).

Editors only open the top of the queue. Precision@K asks: of the K pages we ranked first, what share truly belonged in the review set (proxy today; observed future decline later)? That matches the decision cost ÔÇö wasted editor hours on bad tops, or missing a high-visibility slide.

Also track:
- **recall@K** among high-impression pages (did we miss urgent ones?);
- a **hand review of the top 20** (reason codes must make sense to a human).

"Good" for this phase: a transparent baseline beats random ordering on precision@20, and the top-20 reasons are inspectable.

In [4]:
SUCCESS_METRIC = "precision@K (K=20, K=50)"
GOOD_MEANS = "Top-K pages are mostly true review candidates; beats random / simple volume sort"
SECONDARY = ["recall@K on high-impression pages", "human sense-check of top-20 reason codes"]
print(f"Primary success metric: {SUCCESS_METRIC}")
print(f"What 'good' means: {GOOD_MEANS}")
print("Also track:", "; ".join(SECONDARY))

Primary success metric: precision@K (K=20, K=50)
What 'good' means: Top-K pages are mostly true review candidates; beats random / simple volume sort
Also track: recall@K on high-impression pages; human sense-check of top-20 reason codes


## 4. The unit of analysis, as a real dataframe

**One row = one content item (one anonymized page / `content_id`).**

Lane-2 slice for ranking practice: keep pages with meaningful search demand (`impressions_90d >= 500`). Low-volume rows are mostly noise for a refresh queue. The code below loads that slice, prints shape, and adds a sketched `needs_refresh_review` proxy column plus a toy `priority_score` so the future ranked output is concrete.

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Lane 2 working unit: one content item with enough demand to justify review time
MIN_IMPRESSIONS = 500
unit = df.loc[df["impressions_90d"] >= MIN_IMPRESSIONS].copy()

# Sketched TARGET PROXY (starter only) ÔÇö observed future label comes later from warehouse windows
unit["needs_refresh_review"] = (
    (unit["trend_direction"] == "down").astype(int)
)

# Toy PRIORITY SCORE for framing (not a trained model): demand + decline signal + weak freshness
# Safe-ish inputs for a *baseline sketch*; trend_* stay out of feature lists in later weeks
imp = np.log1p(unit["impressions_90d"])
imp = (imp - imp.min()) / (imp.max() - imp.min() + 1e-9)
stale = (unit["days_since_last_update"].fillna(0) >= 90).astype(float)
page1 = (unit["position_tier"] == "page_1").astype(float)
unit["priority_score"] = (
    0.45 * unit["needs_refresh_review"]
    + 0.35 * imp
    + 0.10 * page1
    + 0.10 * stale
)

show_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "position_tier",
    "days_since_last_update",
    "trend_direction",
    "needs_refresh_review",
    "priority_score",
]

print("Unit of analysis: ONE ROW = ONE content item (page)")
print(f"Full starter rows: {len(df):,}")
print(f"Lane-2 slice (impressions_90d >= {MIN_IMPRESSIONS}): {len(unit):,} rows")
print(f"Proxy positive rate (needs_refresh_review=1): {unit['needs_refresh_review'].mean():.1%}")
print()
print("Sample of the unit dataframe (top by sketched priority_score):")
display_df = unit[show_cols].sort_values("priority_score", ascending=False).head(8)
display_df

Unit of analysis: ONE ROW = ONE content item (page)
Full starter rows: 30,000
Lane-2 slice (impressions_90d >= 500): 16,726 rows
Proxy positive rate (needs_refresh_review=1): 59.6%

Sample of the unit dataframe (top by sketched priority_score):
          content_id         client_id  impressions_90d position_tier  days_since_last_update trend_direction  needs_refresh_review  priority_score
content_5fe46e04994d client_4e07408562           517715        page_1                     104            down                     1        1.000000
content_2c2606c5d176 client_19581e27de           347399        page_1                     104            down                     1        0.979882
content_cb112fce36be client_19581e27de           309910        page_1                     104            down                     1        0.974123
content_c8e9d6ab9013 client_19581e27de           208678        page_1                     104            down                     1        0.954180
content_3d94572

## 5. Why ML beats a fixed rule here

A single if-statement (`if trending down ÔåÆ refresh`) fails because:

1. **Many signals interact** ÔÇö impressions, position tier, freshness, engagement, content type, and client mix all change who deserves the next hour of editor time.
2. **False positives are expensive** ÔÇö declining low-demand pages can flood a naive queue and waste edits that never matter.
3. **False negatives are expensive too** ÔÇö page-one URLs with soft decline can look "fine" under a blunt rule and keep sliding.
4. **A ranked score + reason codes** turns tangled evidence into an action list (refresh, expand, protect, prune, monitor) an editor can challenge.

ML/analysis earns its place as **decision support**: better triage than a fixed rule, not as "predicting Google" or causal proof that a refresh will win traffic back.

In [6]:
# Compare a blunt rule vs demand-aware triage (illustration only ÔÇö not a trained model)
blunt = (df["trend_direction"] == "down").sum()
lane_slice_pos = int(unit["needs_refresh_review"].sum())
high_imp_not_down = int(((df["impressions_90d"] >= MIN_IMPRESSIONS) & (df["trend_direction"] != "down")).sum())

print(f"Blunt rule flags (all down-trend rows): {blunt:,}")
print(f"Lane-2 queue size (high-imp AND down): {lane_slice_pos:,}")
print(f"High-imp pages NOT down (still need ranking vs watchlist): {high_imp_not_down:,}")
print()
print("Point: a fixed 'all down' rule is huge and mixes noise with priority;")
print("scoring lets editors work the top-K of a smaller, demand-aware set.")
print()
print("One-paragraph ML frame:")
print(
    "For an SEO/content editor deciding which pages to review this week, "
    "we will build a ranked priority score from content and search signals, "
    "using a starter proxy (high-demand + downward trend) and later an observed "
    "next-30d decline label, measured by precision@K. A wrong call wastes editor "
    "hours or misses a high-visibility slide. A plain rule is not enough because "
    "demand, position, freshness, and movement interact. We claim only "
    "decision-support / directional results ÔÇö not causal recovery or Google prediction."
)

Blunt rule flags (all down-trend rows): 16,262
Lane-2 queue size (high-imp AND down): 9,961
High-imp pages NOT down (still need ranking vs watchlist): 6,765

Point: a fixed 'all down' rule is huge and mixes noise with priority;
scoring lets editors work the top-K of a smaller, demand-aware set.

One-paragraph ML frame:
For an SEO/content editor deciding which pages to review this week, we will build a ranked priority score from content and search signals, using a starter proxy (high-demand + downward trend) and later an observed next-30d decline label, measured by precision@K. A wrong call wastes editor hours or misses a high-visibility slide. A plain rule is not enough because demand, position, freshness, and movement interact. We claim only decision-support / directional results ÔÇö not causal recovery or Google prediction.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled ÔÇö markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime ÔåÆ Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` ÔÇö then submit your repo URL on the card. Done.

**Colab:** open the badge above ÔåÆ Runtime ÔåÆ Run all ÔåÆ File ÔåÆ Save a copy in GitHub (path `work/notebooks/w02_ml_task_framing.ipynb`) so outputs are visible on github.com.